# QuantJourney SDK - Spread and Liquidity Monitoring Packet

This notebook demonstrates a QuantJourney SDK workflow that builds spread, liquidity, slippage and capacity diagnostics from intraday prices, adjusted OHLCV, volume, short interest and volatility context.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

## Imports and Plot Style

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})


## QuantJourney Client

In [ ]:
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2020-01-01')
END = os.getenv('QJ_EXAMPLE_END') or pd.Timestamp.today().normalize().strftime('%Y-%m-%d')


## Response Helpers

In [ ]:
def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ('rows', 'data', 'items', 'prices', 'results'):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


## Market Data Helpers

In [ ]:
def price_frame(symbol: str, start: str=START, end: str=END) -> pd.DataFrame:
    payload = qj.eod.get_historical_prices(symbol=symbol, start_date=start, end_date=end)
    rows = as_rows(payload)
    if not rows and isinstance(unwrap(payload), dict):
        value = unwrap(payload)
        rows = value.get(symbol) or value.get(symbol.upper()) or []
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'No price data returned for {symbol}')
    df['date'] = pd.to_datetime(df['date'])
    for col in ['open', 'high', 'low', 'close', 'adjusted_close', 'volume']:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'adjusted_close' in df and df['adjusted_close'].notna().any():
        df['price'] = df['adjusted_close'].fillna(df['close'])
    else:
        df['price'] = df['close']
    if 'volume' not in df:
        df['volume'] = np.nan
    return df.dropna(subset=['price']).sort_values('date').set_index('date')

def price_panel(symbols: list[str], start: str=START, end: str=END) -> tuple[pd.DataFrame, pd.DataFrame]:
    prices = {}
    volumes = {}
    for symbol in symbols:
        df = price_frame(symbol, start=start, end=end)
        prices[symbol] = df['price']
        volumes[symbol] = df['volume']
    return (pd.DataFrame(prices).dropna(how='all'), pd.DataFrame(volumes).reindex(pd.DataFrame(prices).index))

def returns(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how='all')

def dollar_adv(prices: pd.DataFrame, volumes: pd.DataFrame, window: int=63) -> pd.DataFrame:
    return (prices * volumes).rolling(window).mean()


In [ ]:
symbols = ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL', 'META']
intraday_raw = {symbol: qj.eod.get_intraday_prices(symbol=symbol, interval='5m', from_date=START, to_date=END) for symbol in symbols[:3]}
live_raw = qj.fmp.get_live_prices(symbol=','.join(symbols))
short_raw = {symbol: qj.finra.get_short_interest(symbol=symbol) for symbol in symbols[:4]}
vix_raw = qj.cboe.get_vix_data(start_date='2023-01-01', end_date=END)
prices, volumes = price_panel(symbols, start='2023-01-01', end=END)


In [ ]:
ret = returns(prices)
adv = dollar_adv(prices, volumes).iloc[-1]
realized_vol = ret.tail(63).std() * np.sqrt(252)
amihud = (ret.abs() / (prices * volumes).replace(0, np.nan)).tail(63).mean()
short_rows = pd.Series({symbol: len(as_rows(payload)) for symbol, payload in short_raw.items()}, name='short_feed_rows')
spread_proxy_bps = (realized_vol.rank(pct=True) * 12 + (1 / adv.rank(pct=True)).replace(np.inf, np.nan) * 4).rename('spread_proxy_bps')
liquidity = pd.DataFrame({'adv_usd': adv, 'realized_vol_63d': realized_vol, 'amihud_proxy': amihud, 'spread_proxy_bps': spread_proxy_bps, 'sl_to_adv_ratio': spread_proxy_bps / adv.rank(pct=True), 'short_feed_rows': short_rows}).sort_values('sl_to_adv_ratio', ascending=False)


In [ ]:
order_notional = 25000000
liquidity['days_at_5pct_adv'] = order_notional / (liquidity['adv_usd'] * 0.05)
liquidity['slippage_cost_bps'] = liquidity['spread_proxy_bps'] * np.sqrt(np.maximum(liquidity['days_at_5pct_adv'], 0.1))
display(pd.Series({'live_price_rows': len(as_rows(live_raw)), 'vix_rows': len(as_rows(vix_raw)), 'intraday_feeds': len(intraday_raw)}))
display(liquidity)
liquidity[['spread_proxy_bps', 'sl_to_adv_ratio', 'days_at_5pct_adv', 'slippage_cost_bps']].plot(kind='bar', subplots=True, layout=(2, 2), figsize=(14, 7), title='Spread and liquidity diagnostics')
plt.tight_layout()
plt.show()


## Notes

This is an example workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.